# 02 — Model Training
**Sign2Chat — UAE Sign Language Recognition**

Trains a CNN + BiLSTM model on the extracted landmark sequences.

### Input
- `landmarks_index_augmented.csv` from notebook 01
- `.npy` files shape `(180, 165)` — 6 seconds @ 30fps, 55 nodes × 3 coords

### Architecture
```
Input (180, 165)
  → Masking                    ignore zero-padded frames
  → Conv1D(64,  k=5) + BN + Dropout(0.4)
  → Conv1D(128, k=5) + BN + Dropout(0.4)
  → Bidirectional LSTM(128)    + BN + Dropout(0.4)
  → Dense(256, relu, L2)       + BN + Dropout(0.4)
  → Dense(NUM_CLASSES, softmax)
```

### Key decisions
- Label smoothing 0.1 — prevents overconfidence
- Gradient clipping 1.0 — stable training
- Early stopping patience=20, restore best weights
- Class weights — handles any class imbalance
- ReduceLROnPlateau — adapts learning rate automatically

### Output files
| File | Description |
|---|---|
| `models/uaesl_best.keras` | Best model checkpoint |
| `models/uaesl_class_index.json` | class_index (0..N-1) → label_id → clean_name |
| `models/uaesl_training_history.png` | Loss and accuracy curves |
| `models/uaesl_results.json` | Final val/test metrics |

In [2]:
!pip install tensorflow pandas numpy matplotlib scikit-learn tqdm

In [2]:
# Must run BEFORE importing TensorFlow
import os
os.environ["XLA_FLAGS"]         = "--xla_gpu_autotune_level=0"
os.environ["TF_XLA_FLAGS"]      = "--tf_xla_auto_jit=0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print("✅ XLA autotuning disabled")

✅ XLA autotuning disabled


## Step 1 — Imports & Config

In [3]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Masking, TimeDistributed, BatchNormalization, Dropout,
    Bidirectional, LSTM, Dense
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint,
    ReduceLROnPlateau, Callback
)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# ── PATHS ─────────────────────────────────────────────────────────────────────
INDEX_CSV      = '../UAE-dataset/landmarks_index_augmented.csv'
CLASS_MAP_PATH = '../UAE-dataset/class_map.json'
MODEL_DIR      = '../models'
BEST_MODEL     = os.path.join(MODEL_DIR, 'uaesl_best.keras')
CLASS_IDX_PATH = os.path.join(MODEL_DIR, 'uaesl_class_index.json')
HISTORY_PNG    = os.path.join(MODEL_DIR, 'uaesl_training_history.png')
RESULTS_PATH   = os.path.join(MODEL_DIR, 'uaesl_results.json')

os.makedirs(MODEL_DIR, exist_ok=True)

# ── Sequence config — must match notebook 01 ──────────────────────────────────
MAX_FRAMES  = 180
FEATURE_DIM = 165

# ── Training hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE      = 16
EPOCHS          = 200
LEARNING_RATE   = 5e-4
L2_REG          = 1e-3
DROPOUT         = 0.4
LABEL_SMOOTHING = 0.1
PATIENCE        = 20     # early stopping
RANDOM_SEED     = 42

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# GPU config
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print(f'✅ TensorFlow {tf.__version__}')
print(f'   GPU available : {len(gpus) > 0}')
if gpus:
    print(f'   GPU           : {gpus[0].name}')
print(f'   MAX_FRAMES    : {MAX_FRAMES}')
print(f'   FEATURE_DIM   : {FEATURE_DIM}')

I0000 00:00:1781302248.967242  141665 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781302250.235938  141665 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781302253.164447  141665 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ TensorFlow 2.21.0
   GPU available : True
   GPU           : /physical_device:GPU:0
   MAX_FRAMES    : 180
   FEATURE_DIM   : 165


In [4]:
import tensorflow as tf
print(tf.__version__)
print(tf.sysconfig.get_build_info()['cuda_version'])
print(tf.sysconfig.get_build_info()['cudnn_version'])

# Also check if GPU is actually visible
print(tf.config.list_physical_devices('GPU'))

2.21.0
12.5.1
9
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Step 2 — Load Index & Build Class Mapping

In [6]:
df = pd.read_csv(INDEX_CSV)

print(f'✅ Index loaded: {len(df)} total samples')
print(df['split'].value_counts().to_string())

with open(CLASS_MAP_PATH) as f:
    class_map = json.load(f)

present_label_ids = sorted(df['label_id'].unique().tolist())  # ← .tolist() converts int64 → int
NUM_CLASSES       = len(present_label_ids)
label_id_to_idx   = {lid: idx for idx, lid in enumerate(present_label_ids)}

idx_to_info = {
    idx: {
        'label_id':   int(lid),                           # ← explicit int()
        'clean_name': class_map[str(lid)]['clean_name'],
        'word':       class_map[str(lid)]['word'],
    }
    for lid, idx in label_id_to_idx.items()
}

# Save — all values are now native Python ints, JSON can serialize them
with open(CLASS_IDX_PATH, 'w') as f:
    json.dump({str(k): v for k, v in idx_to_info.items()}, f, indent=2)

print(f'\n✅ Class mapping built')
print(f'   NUM_CLASSES : {NUM_CLASSES}')
print(f'   Sample: class 0 → label_id={idx_to_info[0]["label_id"]} → {idx_to_info[0]["clean_name"]}')
print(f'   Saved  → {CLASS_IDX_PATH}')

✅ Index loaded: 32341 total samples
split
train    31053
test       665
val        623

✅ Class mapping built
   NUM_CLASSES : 100
   Sample: class 0 → label_id=34 → Cat
   Saved  → ../models/uaesl_class_index.json


## Step 3 — Load All .npy Files into Memory

In [7]:
def load_split(df, split):
    """Load all .npy files for a given split into X, y arrays."""
    rows = df[df['split'] == split].reset_index(drop=True)
    X    = np.zeros((len(rows), MAX_FRAMES, FEATURE_DIM), dtype=np.float32)
    y    = np.zeros(len(rows), dtype=np.int32)

    missing = []
    for i, row in rows.iterrows():
        path = row['npy_path']
        if not os.path.exists(path):
            missing.append(path)
            continue
        seq = np.load(path)
        # Safety check — reshape if needed
        if seq.shape != (MAX_FRAMES, FEATURE_DIM):
            if seq.shape[1] == FEATURE_DIM:
                T = seq.shape[0]
                if T >= MAX_FRAMES:
                    seq = seq[-MAX_FRAMES:]
                else:
                    pad = np.zeros((MAX_FRAMES - T, FEATURE_DIM), dtype=np.float32)
                    seq = np.vstack([pad, seq])
            else:
                missing.append(path)
                continue
        X[i] = seq
        y[i] = label_id_to_idx[row['label_id']]

    if missing:
        print(f'  ⚠️  {len(missing)} missing files in {split} split')

    return X, y


print('Loading splits into memory ...')
X_train, y_train = load_split(df, 'train')
X_val,   y_val   = load_split(df, 'val')
X_test,  y_test  = load_split(df, 'test')

# One-hot encode labels
y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_cat   = tf.keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  NUM_CLASSES)

print(f'\n✅ Data loaded')
print(f'   X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'   X_val   : {X_val.shape}   y_val   : {y_val.shape}')
print(f'   X_test  : {X_test.shape}   y_test  : {y_test.shape}')
print(f'\n   Value range: [{X_train.min():.3f}, {X_train.max():.3f}]')
print(f'   Non-zero train frames avg: {np.any(X_train != 0, axis=2).sum(axis=1).mean():.1f} / {MAX_FRAMES}')

Loading splits into memory ...

✅ Data loaded
   X_train : (31053, 180, 165)   y_train : (31053,)
   X_val   : (623, 180, 165)   y_val   : (623,)
   X_test  : (665, 180, 165)   y_test  : (665,)

   Value range: [-2.731, 2.383]
   Non-zero train frames avg: 149.7 / 180


## Step 4 — Class Weights

In [8]:
# Compute class weights to handle any class imbalance
# Classes with fewer training samples get higher weight
classes    = np.unique(y_train)
weights    = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))

print('✅ Class weights computed')
print(f'   Min weight : {min(weights):.3f}')
print(f'   Max weight : {max(weights):.3f}')
print(f'   Mean weight: {weights.mean():.3f}')
if max(weights) / min(weights) > 2:
    print(f'\n⚠️  Large weight spread ({max(weights)/min(weights):.1f}x) — some classes have significantly fewer samples')
else:
    print(f'\n   Weight spread is small ({max(weights)/min(weights):.1f}x) — classes are balanced ✅')

✅ Class weights computed
   Min weight : 0.743
   Max weight : 1.344
   Mean weight: 1.038

   Weight spread is small (1.8x) — classes are balanced ✅


## Step 5 — Build Model

### Architecture rationale
- **Masking** — ignores zero-padded frames at the front of each sequence
- **Conv1D ×2** — extracts local motion patterns (5-frame windows = 0.17s)
- **Bidirectional LSTM** — captures temporal dependencies in both directions
- **Dense head** — final classification with L2 regularization
- **Label smoothing** — prevents the model from becoming overconfident
- **Gradient clipping** — stabilizes training, prevents exploding gradients

In [8]:
def build_model(num_classes, max_frames, feature_dim):
    model = Sequential([
        # ── Input & masking ────────────────────────────────────────────────
        Masking(mask_value=0.0, input_shape=(max_frames, feature_dim)),

        # ── Per-frame feature extraction (TimeDistributed Dense) ──────────
        # Uses matrix-multiply kernels, not cuDNN Conv — avoids cuDNN 9/TF 2.21
        # CUDNN_BACKEND_TENSOR_DESCRIPTOR bug on this GPU.
        TimeDistributed(Dense(128, activation='relu')),
        BatchNormalization(),
        Dropout(DROPOUT),

        TimeDistributed(Dense(128, activation='relu')),
        BatchNormalization(),
        Dropout(DROPOUT),

        # ── Bidirectional LSTM — full sequence context ─────────────────────
        Bidirectional(LSTM(128, return_sequences=False, use_cudnn=False)),
        BatchNormalization(),
        Dropout(DROPOUT),

        # ── Dense head ────────────────────────────────────────────────────
        Dense(256, activation='relu', kernel_regularizer=l2(L2_REG)),
        BatchNormalization(),
        Dropout(DROPOUT),

        # ── Output ────────────────────────────────────────────────────────
        Dense(num_classes, activation='softmax', dtype='float32',
              kernel_regularizer=l2(L2_REG)),
    ])

    model.compile(
        optimizer = tf.keras.optimizers.Adam(
            learning_rate = LEARNING_RATE,
            clipnorm      = 1.0
        ),
        loss    = tf.keras.losses.CategoricalCrossentropy(
            label_smoothing = LABEL_SMOOTHING
        ),
        metrics = [
            'accuracy',
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
        ]
    )
    return model


model = build_model(NUM_CLASSES, MAX_FRAMES, FEATURE_DIM)
model.summary()

total_params = model.count_params()
print(f'\n✅ Model ready')
print(f'   Parameters   : {total_params:,}')
print(f'   Input shape  : (None, {MAX_FRAMES}, {FEATURE_DIM})')
print(f'   Output shape : (None, {NUM_CLASSES})')
print(f'   Label smooth : {LABEL_SMOOTHING}')
print(f'   Grad clip    : 1.0')


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 180, 165)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 180, 128)       │        21,248 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 180, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 180, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 180, 128)       │        16,512 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 180, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 180, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │        25,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 395,492 (1.51 MB)

 Trainable params: 393,956 (1.50 MB)

 Non-trainable params: 1,536 (6.00 KB)


✅ Model ready
   Parameters   : 395,492
   Input shape  : (None, 180, 165)
   Output shape : (None, 100)
   Label smooth : 0.1
   Grad clip    : 1.0


## Step 6 — Callbacks

In [9]:
class OverfitGuard(Callback):
    """Stops training when train_acc - val_acc gap exceeds max_gap.
    Saves time when the model is clearly memorizing training data.
    """
    def __init__(self, max_gap=0.25, patience=3):
        super().__init__()
        self.max_gap    = max_gap
        self.patience   = patience
        self._count     = 0

    def on_epoch_end(self, epoch, logs=None):
        train_acc = logs.get('accuracy',     0)
        val_acc   = logs.get('val_accuracy', 0)
        gap       = train_acc - val_acc
        if gap > self.max_gap:
            self._count += 1
            if self._count >= self.patience:
                print(f'\n⚠️  OverfitGuard: gap {gap*100:.1f}% > '
                      f'{self.max_gap*100:.0f}% for {self.patience} epochs — stopping')
                self.model.stop_training = True
        else:
            self._count = 0


callbacks = [
    # Save best model by val_accuracy
    ModelCheckpoint(
        filepath          = BEST_MODEL,
        monitor           = 'val_accuracy',
        save_best_only    = True,
        save_weights_only = False,
        verbose           = 1
    ),

    # Stop when val_accuracy stops improving
    EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = PATIENCE,
        restore_best_weights = True,
        verbose              = 1
    ),

    # Halve learning rate when val_loss plateaus
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 8,
        min_lr   = 1e-7,
        verbose  = 1
    ),

    # Stop early if train/val gap exceeds 25%
    OverfitGuard(max_gap=0.25, patience=3),
]

print('✅ Callbacks ready')
print(f'   ModelCheckpoint → {BEST_MODEL}')
print(f'   EarlyStopping   patience={PATIENCE}')
print(f'   ReduceLROnPlateau  factor=0.5, patience=8')
print(f'   OverfitGuard    max_gap=25%, patience=3')

✅ Callbacks ready
   ModelCheckpoint → ../models/uaesl_best.keras
   EarlyStopping   patience=20
   ReduceLROnPlateau  factor=0.5, patience=8
   OverfitGuard    max_gap=25%, patience=3


## Step 7 — Train

In [10]:
# ── GPU smoke test: run this first ────────────────────────────────────────────
import numpy as _np
_tiny_x = _np.random.randn(4, MAX_FRAMES, FEATURE_DIM).astype('float32')
_tiny_y = tf.keras.utils.to_categorical(_np.array([0, 1, 2, 3]), NUM_CLASSES)

_tiny = tf.keras.Sequential([
    tf.keras.layers.Masking(input_shape=(MAX_FRAMES, FEATURE_DIM)),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(16, activation='relu')),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16, use_cudnn=False)),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32'),
])
_tiny.compile(optimizer='adam', loss='categorical_crossentropy')
_tiny.fit(_tiny_x, _tiny_y, epochs=2, verbose=0)
print('✅ GPU smoke test passed — safe to run full training')
del _tiny, _tiny_x, _tiny_y


✅ GPU smoke test passed — safe to run full training


In [ ]:
print(f'Training on {len(X_train)} samples, validating on {len(X_val)} ...')
print(f'Batch size : {BATCH_SIZE}')
print(f'Max epochs : {EPOCHS}')
print()

history = model.fit(
    X_train, y_train_cat,
    validation_data = (X_val, y_val_cat),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weight_dict,
    callbacks       = callbacks,
    shuffle         = True,
    verbose         = 1,
)

best_epoch   = int(np.argmax(history.history['val_accuracy'])) + 1
best_val_acc = max(history.history['val_accuracy'])
best_top5    = history.history['val_top5_acc'][best_epoch - 1]
total_epochs = len(history.history['accuracy'])
train_acc_at_best = history.history['accuracy'][best_epoch - 1]

print(f'\n✅ Training complete')
print(f'   Epochs run        : {total_epochs}')
print(f'   Best epoch        : {best_epoch}')
print(f'   Best val_accuracy : {best_val_acc*100:.2f}%')
print(f'   Best val_top5_acc : {best_top5*100:.2f}%')
print(f'   Train acc at best : {train_acc_at_best*100:.2f}%')
print(f'   Overfit gap       : {(train_acc_at_best - best_val_acc)*100:.1f}%')

Training on 31053 samples, validating on 623 ...
Batch size : 16
Max epochs : 200

Epoch 1/200
1941/1941 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0443 - loss: 5.2332 - top5_acc: 0.1500
Epoch 1: val_accuracy improved from None to 0.19422, saving model to ../models/uaesl_best.keras
1941/1941 ━━━━━━━━━━━━━━━━━━━━ 2473s 1s/step - accuracy: 0.0895 - loss: 4.6780 - top5_acc: 0.2636 - val_accuracy: 0.1942 - val_loss: 3.7905 - val_top5_acc: 0.5281 - learning_rate: 5.0000e-04
Epoch 2/200
1941/1941 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2795 - loss: 3.4229 - top5_acc: 0.6266
Epoch 2: val_accuracy improved from 0.19422 to 0.62921, saving model to ../models/uaesl_best.keras
1941/1941 ━━━━━━━━━━━━━━━━━━━━ 2446s 1s/step - accuracy: 0.3572 - loss: 3.1357 - top5_acc: 0.7126 - val_accuracy: 0.6292 - val_loss: 2.3718 - val_top5_acc: 0.9037 - learning_rate: 5.0000e-04
Epoch 3/200
1941/1941 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5551 - loss: 2.4717 - top5_acc: 0.8739
Epoch 3: val_accura

## Step 8 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('UAE SL — CNN + BiLSTM Training History', fontsize=13)

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs_range, history.history['accuracy'],    label='Train')
axes[0].plot(epochs_range, history.history['val_accuracy'],label='Val')
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best (e{best_epoch})')
axes[0].set_title('Top-1 Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Top-5 Accuracy
axes[1].plot(epochs_range, history.history['top5_acc'],    label='Train')
axes[1].plot(epochs_range, history.history['val_top5_acc'],label='Val')
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Top-5 Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Loss
axes[2].plot(epochs_range, history.history['loss'],    label='Train')
axes[2].plot(epochs_range, history.history['val_loss'],label='Val')
axes[2].axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('Loss')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(HISTORY_PNG, dpi=150)
plt.show()
print(f'✅ Saved → {HISTORY_PNG}')

## Step 9 — Evaluate on Test Set

In [ ]:
# Load best saved model
best_model = tf.keras.models.load_model(BEST_MODEL)
print(f'✅ Loaded best model from {BEST_MODEL}')

# Evaluate on test set
test_loss, test_acc, test_top5 = best_model.evaluate(
    X_test, y_test_cat, batch_size=BATCH_SIZE, verbose=0)

# Get predictions for top-10
y_pred_probs = best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

top1  = (y_pred == y_test).mean()
top5  = sum(y_test[i] in np.argsort(y_pred_probs[i])[-5:]
            for i in range(len(y_test))) / len(y_test)
top10 = sum(y_test[i] in np.argsort(y_pred_probs[i])[-10:]
            for i in range(len(y_test))) / len(y_test)

print(f'\n=== Test Set Results ===')
print(f'   Top-1  : {top1*100:.2f}%')
print(f'   Top-5  : {top5*100:.2f}%')
print(f'   Top-10 : {top10*100:.2f}%')

## Step 10 — Per-Class Analysis

In [ ]:
per_class = {}
for idx in range(NUM_CLASSES):
    mask = y_test == idx
    if mask.sum() == 0:
        continue
    info = idx_to_info[idx]
    acc  = (y_pred[mask] == idx).mean()
    per_class[info['clean_name']] = {
        'class_idx': idx,
        'label_id':  info['label_id'],
        'acc':       round(float(acc) * 100, 1),
        'n_test':    int(mask.sum())
    }

sorted_acc = sorted(per_class.items(), key=lambda x: x[1]['acc'])

print(f'  {"Class":<25} {"Acc":>6}   {"Test samples":>12}')
print(f'  {"-"*25} {"-"*6}   {"-"*12}')
print('  --- Bottom 10 ---')
for g, d in sorted_acc[:10]:
    print(f'  {g:<25} {d["acc"]:>5.1f}%   {d["n_test"]:>12}')
print('  --- Top 10 ---')
for g, d in sorted_acc[-10:][::-1]:
    print(f'  {g:<25} {d["acc"]:>5.1f}%   {d["n_test"]:>12}')

## Step 11 — Save Results

In [ ]:
results = {
    'model':        'CNN + BiLSTM — UAE Sign Language',
    'num_classes':  NUM_CLASSES,
    'max_frames':   MAX_FRAMES,
    'feature_dim':  FEATURE_DIM,
    'training': {
        'total_epochs':   total_epochs,
        'best_epoch':     best_epoch,
        'best_val_top1':  round(best_val_acc * 100, 2),
        'best_val_top5':  round(best_top5    * 100, 2),
        'overfit_gap':    round((train_acc_at_best - best_val_acc) * 100, 1),
    },
    'test': {
        'top1':  round(top1  * 100, 2),
        'top5':  round(top5  * 100, 2),
        'top10': round(top10 * 100, 2),
        'n_samples': int(len(y_test)),
    },
    'per_class': per_class,
}

with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f'✅ Results saved → {RESULTS_PATH}')
print(f'\n{"="*50}')
print(f'TRAINING COMPLETE')
print(f'{"="*50}')
print(f'  Model      : CNN + BiLSTM')
print(f'  Classes    : {NUM_CLASSES}')
print(f'  Best epoch : {best_epoch} / {total_epochs}')
print(f'\n  Validation (best epoch):')
print(f'    Top-1 : {best_val_acc*100:.2f}%')
print(f'    Top-5 : {best_top5*100:.2f}%')
print(f'\n  Test set:')
print(f'    Top-1 : {top1*100:.2f}%')
print(f'    Top-5 : {top5*100:.2f}%')
print(f'    Top-10: {top10*100:.2f}%')
print(f'\n  Saved files:')
print(f'    {BEST_MODEL}')
print(f'    {CLASS_IDX_PATH}')
print(f'    {HISTORY_PNG}')
print(f'    {RESULTS_PATH}')
print(f'\n✅ Ready for notebook 03 — evaluation & live demo')